# BSCP sack-train-ml — Train a Run

## ▶ To start: click **Runtime → Run all** (or press `Cmd/Ctrl + F9`)

This notebook trains a YOLO model for a registry run id and streams metrics + final artifacts back to Supabase.

**Before you Run all, check:**
1. **GPU runtime is set** — Runtime → Change runtime type → T4 (free) or better.
2. **You have a Supabase service-role key** — the notebook will prompt for it.
3. **You have the training callback HMAC secret** — the notebook will prompt for it.
4. **Run id is in the URL** as `?run_id=...`. If not, paste it manually when prompted.
5. **The setup cell prints a git SHA** after syncing `main`; the training log should show the same SHA before export starts.

## 1. Resolve the run id from the URL

In [ ]:
from urllib.parse import urlparse, parse_qs

RUN_ID = ''
try:
    from google.colab import _message  # type: ignore
    raw = _message.blocking_request('get_url', timeout_sec=5)
    # Newer Colab returns {'url': '...'}; older returns the URL string directly.
    url = raw.get('url', '') if isinstance(raw, dict) else (raw or '')
    qs = parse_qs(urlparse(url).query)
    RUN_ID = qs.get('run_id', [''])[0]
except Exception as exc:
    print('Could not auto-read run_id from the URL:', exc)

print('Run id:', RUN_ID or '(none — paste below)')


In [ ]:
if not RUN_ID:
    RUN_ID = input('Paste run id: ').strip()
assert RUN_ID, 'run_id is required'
import os
os.environ['BSCP_RUN_ID'] = RUN_ID

## 2. Install dependencies + sync the repo

In [ ]:
%pip install --quiet ultralytics onnx onnxsim pyyaml numpy 2>&1 | tail -3

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/pitikorn-pam/sack-train-ml.git'
REPO_DIR = Path('/content/sack-train-ml')
BRANCH = 'main'

if REPO_DIR.exists():
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

GIT_SHA = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, text=True).strip()
os.environ['BSCP_GIT_SHA'] = GIT_SHA
print('Repo synced at', GIT_SHA)

# Editable install — show stderr if anything goes wrong
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR)],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('--- pip stdout ---')
    print(result.stdout[-2000:])
    print('--- pip stderr ---')
    print(result.stderr[-2000:])
    raise SystemExit(f'pip install -e failed (exit {result.returncode})')

# Belt-and-braces: also add src/ to sys.path so the import works even if the
# editable .pth file isn't picked up by this kernel session.
src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Smoke-test the import right here so failures surface in this cell, not the next.
import importlib
if 'sack_train_ml' in sys.modules:
    importlib.reload(sys.modules['sack_train_ml'])
import sack_train_ml  # noqa: F401

print('sack_train_ml installed + importable')


## 3. Supabase auth (paste service-role + callback secret)

In [ ]:
from getpass import getpass

SUPABASE_URL = input('SUPABASE_URL (e.g. https://xxxxx.supabase.co): ').strip()
SUPABASE_SERVICE_ROLE_KEY = getpass('SUPABASE_SERVICE_ROLE_KEY: ')
TRAINING_CALLBACK_SECRET = getpass('TRAINING_CALLBACK_SECRET: ')

assert SUPABASE_URL.startswith('https://') and '.supabase.co' in SUPABASE_URL, 'invalid SUPABASE_URL'
assert SUPABASE_SERVICE_ROLE_KEY.startswith('eyJ'), 'service_role_key looks malformed (should be a JWT)'
assert len(TRAINING_CALLBACK_SECRET) >= 32, 'callback secret looks too short'

import os
os.environ['SUPABASE_URL'] = SUPABASE_URL
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = SUPABASE_SERVICE_ROLE_KEY
os.environ['TRAINING_CALLBACK_SECRET'] = TRAINING_CALLBACK_SECRET
print('Auth env set.')

## 4. Connectivity check

In [ ]:
import os
from sack_train_ml.supabase_client import RegistryClient

client = RegistryClient()
run = client.fetch_run(os.environ['BSCP_RUN_ID'])
print('Run status:', run['status'])
print('Config keys:', list((run.get('config_yaml') or {}).keys()))

client.log_step(os.environ['BSCP_RUN_ID'], 1, 'init', 'info',
                f"colab notebook attached · git={os.environ.get('BSCP_GIT_SHA', '?')}")

## 5. Run the training pipeline

Calls `scripts/train_for_run.py`, which orchestrates every stage and streams metrics live to the dashboard via the `training-callback` edge function.

In [ ]:
import os, subprocess, sys

SKIP_HEF = os.environ.get('SKIP_HEF', '').lower() in ('1', 'true', 'yes')
args = [sys.executable, '/content/sack-train-ml/scripts/train_for_run.py', '--run-id', os.environ['BSCP_RUN_ID']]
if SKIP_HEF:
    args.append('--skip-hef')

rc = subprocess.run(args, check=False).returncode
if rc != 0:
    raise SystemExit(f'train_for_run.py exited {rc}')

## 6. Done

The run row is now `succeeded` (or `failed` with an error log entry). Artifacts are uploaded to R2 and the `versions` row is created. The web dashboard will reflect status live via Supabase Realtime.